# Feature Engineering - Gold Layer (Produção)

Pipeline estável de feature engineering para detecção de fraude.

**Fonte:** `fraud_detection_dev.silver.fraud_data_clean`  
**Destino:** `fraud_detection_dev.gold.fraud_features` (Feature Store)

**Etapas:**
1. Split temporal (60/20/20)
2. Feature selection (qualidade, evidência, redundância)
3. Feature engineering (temporal, email, agregações)
4. Publicação no Unity Catalog Feature Store

**Uso:** Pipeline automatizado de retreino. Para experimentação, use `notebooks/02_FE_PRE_PROCE`.

In [0]:
from pyspark.sql import functions as F, Window
import yaml

# Carregar configuração do projeto
with open('/Workspace/Users/eduardolimaofficial410@gmail.com/databricks-fraud-detection/config/project_config.yml') as f:
    config = yaml.safe_load(f)

FEATURE_TABLE = config['feature_store']['table']
SOURCE_TABLE = 'fraud_detection_dev.silver.fraud_data_clean'

print(f"Source: {SOURCE_TABLE}")
print(f"Target: {FEATURE_TABLE}")

In [0]:
dados = spark.sql(f"SELECT * FROM {SOURCE_TABLE}")
print(f"Registros: {dados.count():,}")
print(f"Colunas: {len(dados.columns)}")

In [0]:
# Split temporal walk-forward
q60, q80 = dados.approxQuantile('TransactionDT', [0.60, 0.80], 0.01)

train_df = dados.filter(F.col('TransactionDT') <= q60)
test_df = dados.filter((F.col('TransactionDT') > q60) & (F.col('TransactionDT') <= q80))
val_df = dados.filter(F.col('TransactionDT') > q80)

print(f"Train: {train_df.count():,}")
print(f"Test: {test_df.count():,}")
print(f"Val: {val_df.count():,}")

In [0]:
# Features selecionadas (resultado da seleção em 3 etapas no notebook de experimentação)
# Etapa 1: Qualidade (71 removidas)
# Etapa 2: Evidência univariada (calculada)
# Etapa 3: Redundância (134 removidas, DeviceType recuperado)

removed_quality = ['C1', 'C2', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D10', 'D11', 'D12', 'D13', 'D14', 'D6', 'D7', 'D8', 'D9', 'M1', 'M2', 'M3', 'V1', 'V10', 'V11', 'V138', 'V14', 'V140', 'V141', 'V142', 'V143', 'V144', 'V145', 'V146', 'V147', 'V148', 'V149', 'V15', 'V150', 'V151', 'V152', 'V153', 'V154', 'V155', 'V156', 'V157', 'V158', 'V159', 'V16', 'V160', 'V161', 'V162', 'V163', 'V164', 'V165', 'V17', 'V18', 'V19', 'V2', 'V3', 'V321', 'V322', 'V323', 'V324', 'V325', 'V326', 'V327', 'V328', 'V329', 'V330', 'V331', 'V332', 'V333', 'V334', 'V335', 'V336', 'V337', 'V338', 'V339', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'id_01', 'id_02']

removed_redundancy = ['C3', 'D15', 'D2', 'D3', 'D4', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'V100', 'V101', 'V102', 'V103', 'V104', 'V105', 'V106', 'V107', 'V108', 'V109', 'V110', 'V111', 'V112', 'V113', 'V114', 'V115', 'V116', 'V117', 'V118', 'V119', 'V12', 'V120', 'V121', 'V122', 'V123', 'V124', 'V125', 'V126', 'V127', 'V128', 'V129', 'V13', 'V130', 'V131', 'V132', 'V133', 'V134', 'V135', 'V136', 'V137', 'V139', 'V166', 'V167', 'V168', 'V169', 'V170', 'V171', 'V172', 'V173', 'V174', 'V175', 'V176', 'V177', 'V178', 'V179', 'V180', 'V181', 'V182', 'V183', 'V184', 'V185', 'V186', 'V187', 'V188', 'V189', 'V190', 'V191', 'V192', 'V193', 'V194', 'V195', 'V196', 'V197', 'V198', 'V199', 'V20', 'V200', 'V201', 'V202', 'V203', 'V204', 'V205', 'V206', 'V207', 'V208', 'V209', 'V21', 'V210', 'V211', 'V212', 'V213', 'V214', 'V215', 'V216', 'V217', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41', 'V42', 'V43', 'V44', 'V68', 'V97', 'V98', 'V99', 'id_12']

features_drop = removed_quality + removed_redundancy + ['ingestion_timestamp']

print(f"Features removidas: {len(features_drop)}")

In [0]:
FREE_DOMAINS = ['gmail.com', 'yahoo.com', 'hotmail.com', 'outlook.com', 'live.com', 'icloud.com', 'aol.com', 'msn.com']

def apply_feature_engineering_spark(df):
    """
    Feature Engineering nativo em PySpark para produção.
    
    Features criadas:
    - Temporais: hora, dia, períodos (manhã/tarde/noite), fim de semana
    - Transação: log(valor), arredondamento
    - Email: match entre P/R, domínios gratuitos
    - Agregações: contagens e estatísticas por card1 e addr1
    """
    # Device
    df = df.withColumn('DeviceType_isnull', F.when(F.col('DeviceType').isNull(), 1).otherwise(0))
    
    # Transação
    df = df.withColumn('TransactionAmt_log', F.log1p(F.col('TransactionAmt')))
    df = df.withColumn('TransactionAmt_round', F.when(F.col('TransactionAmt') % 1 == 0, 1).otherwise(0))
    
    # Temporais
    df = df.withColumn('transaction_hour', ((F.col('TransactionDT') / 3600) % 24).cast('int'))
    df = df.withColumn('transaction_day', ((F.col('TransactionDT') / 86400) % 7).cast('int'))
    df = df.withColumn('is_morning', F.when((F.col('transaction_hour') >= 6) & (F.col('transaction_hour') < 12), 1).otherwise(0))
    df = df.withColumn('is_afternoon', F.when((F.col('transaction_hour') >= 12) & (F.col('transaction_hour') < 18), 1).otherwise(0))
    df = df.withColumn('is_night', F.when((F.col('transaction_hour') >= 18) | (F.col('transaction_hour') < 6), 1).otherwise(0))
    df = df.withColumn('is_weekend', F.when(F.col('transaction_day') >= 5, 1).otherwise(0))
    
    # Email
    df = df.withColumn('email_match', F.when(F.col('P_emaildomain') == F.col('R_emaildomain'), 1).otherwise(0))
    df = df.withColumn('P_email_free', F.when(F.col('P_emaildomain').isin(FREE_DOMAINS), 1).otherwise(0))
    df = df.withColumn('R_email_free', F.when(F.col('R_emaildomain').isin(FREE_DOMAINS), 1).otherwise(0))
    
    # Agregações por card1
    w_card1 = Window.partitionBy('card1')
    df = df.withColumn('card1_tx_count', F.count('*').over(w_card1))
    df = df.withColumn('card1_amt_mean', F.mean('TransactionAmt').over(w_card1))
    df = df.withColumn('card1_amt_std', F.stddev('TransactionAmt').over(w_card1))
    df = df.withColumn('TransactionAmt_dev_card', F.col('TransactionAmt') - F.col('card1_amt_mean'))
    df = df.withColumn('TransactionAmt_zscore_card', F.col('TransactionAmt_dev_card') / (F.col('card1_amt_std') + F.lit(1e-6)))
    
    # Agregações por addr1
    w_addr1 = Window.partitionBy('addr1')
    df = df.withColumn('addr1_tx_count', F.count('*').over(w_addr1))
    df = df.withColumn('addr1_amt_mean', F.mean('TransactionAmt').over(w_addr1))
    
    # Device mismatch (placeholder)
    df = df.withColumn('device_mismatch', F.lit(0))
    
    return df

print("Função de FE definida")

In [0]:
# Remover features descartadas
train_s = train_df.drop(*[c for c in features_drop if c in train_df.columns])
test_s = test_df.drop(*[c for c in features_drop if c in test_df.columns])
val_s = val_df.drop(*[c for c in features_drop if c in val_df.columns])

# Aplicar feature engineering
train_s = apply_feature_engineering_spark(train_s)
test_s = apply_feature_engineering_spark(test_s)
val_s = apply_feature_engineering_spark(val_s)

print(f"Train: {train_s.count():,} | Test: {test_s.count():,} | Val: {val_s.count():,}")
print(f"Colunas após FE: {len(train_s.columns)}")

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

# Criar schema Gold se não existir
uc_parts = FEATURE_TABLE.split(".")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {uc_parts[0]}.{uc_parts[1]}")

# Montar DataFrame único com coluna split para rastreabilidade
full_df = (
    train_s.withColumn("split", F.lit("train"))
    .unionByName(test_s.withColumn("split", F.lit("test")))
    .unionByName(val_s.withColumn("split", F.lit("val")))
    .filter(F.col("TransactionID").isNotNull())
    .withColumn("TransactionDT", F.from_unixtime(F.col("TransactionDT")).cast("timestamp"))
)

row_count = full_df.count()
feature_count = len(full_df.columns) - 4  # Excluir TransactionID, TransactionDT, target, split

# Dropar tabela existente
spark.sql(f"DROP TABLE IF EXISTS {FEATURE_TABLE}")

# Criar feature table com timeseries column
fe = FeatureEngineeringClient()
fe.create_table(
    name=FEATURE_TABLE,
    primary_keys=["TransactionID", "TransactionDT"],
    timeseries_column="TransactionDT",
    df=full_df,
    schema=full_df.schema,
    description="Feature table Gold para detecção de fraude - features engenheiradas em formato RAW, split temporal",
)

# Adicionar comentário na coluna split
spark.sql(
    f"COMMENT ON COLUMN {FEATURE_TABLE}.split IS "
    f"'Conjunto de dados: train, test ou val (split temporal)'"
)

print(f"✅ Feature table publicada: {FEATURE_TABLE}")
print(f"   Linhas: {row_count:,}")
print(f"   Features: {feature_count}")